# Troubleshooting

Things that can go wrong when solving a linopy model, and how to diagnose
them. Each section below is self-contained -- jump to whichever matches your
symptom. This page will grow as more solver issues are documented.

## Infeasible models

From time to time you hit a model that is **infeasible** -- no assignment of
the variables satisfies every constraint at once. Some solvers can report
*which* constraints are in conflict, which is the fastest way to track down
the cause.

linopy exposes this through `compute_infeasibilities` and
`format_infeasibilities`.

We start by creating a small model that is infeasible.

In [ ]:
import pandas as pd

import linopy

m = linopy.Model()

time = pd.RangeIndex(10, name="time")
x = m.add_variables(lower=0, coords=[time], name="x")
y = m.add_variables(lower=0, coords=[time], name="y")

m.add_constraints(x <= 5)
m.add_constraints(y <= 5)
m.add_constraints(x + y >= 12)

# A trivial objective is required; the model is solved purely to check feasibility.
m.add_objective(0 * x)

If we now try to solve the model, we get a message that the model is infeasible.

In [ ]:
m.solve(solver_name="gurobi")

When the solver reports infeasibility, `format_infeasibilities()` runs an IIS (irreducible infeasible subsystem) analysis and returns a human-readable summary of the conflicting constraints. This depends on the solver -- it currently works with `gurobi`, and not every solver supports it.

If you instead need those constraints as data -- to relax or remove them in code -- `compute_infeasibilities()` returns their labels as a list.

In [ ]:
print(m.format_infeasibilities())

## Duplicate Gurobi log lines

When using the Gurobi solver you may see some of the log lines it emits during
the solve duplicated:

```
Total elapsed time = 498.27s
[INFO] Total elapsed time = 498.27s
```

This happens because the Gurobi logger both prints to the console and
propagates to the root logger. Adding the following to your application code
before calling `solve` fixes it:

```python
import logging

logging.getLogger("gurobipy").propagate = False
```

See [this thread](https://groups.google.com/g/gurobi/c/sV7xxN_mzCk) for more.